In [7]:
%load_ext autoreload
%autoreload 2
import sys
import os
sys.path.append(os.path.abspath(".."))  

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
import numpy as np
from models.transformer.model import FeatureTokenizer

np.random.seed(0)

# Giả lập batch nhỏ: 2 mẫu, mỗi mẫu 3 feature (giống 3 cột dữ liệu bảng)
batch, num_features, d_model = 2, 3, 4
x = np.random.randn(batch, num_features)

tok = FeatureTokenizer(num_features=num_features, d_model=d_model)
tokens = tok(x)

print("x shape:", x.shape)            # (2, 3)
print("tokens shape:", tokens.shape)  # (2, 3, 4) -- mỗi feature thành 1 token d_model chiều
print(tokens[0])                      # 3 token của mẫu đầu tiên


x shape: (2, 3)
tokens shape: (2, 3, 4)
[[ 2.37022999 -0.3775979  -0.2575049   1.02433928]
 [ 0.08151537  0.82298465  0.43067715  0.06885683]
 [ 0.61437087  0.4618535   2.06802138 -0.28396869]]


In [12]:
def loss_fn(out):
    return 0.5 * np.sum(out ** 2), out  # dL/dout = out

def numerical_grad(f, arr, eps=1e-4):
    grad = np.zeros_like(arr)
    it = np.nditer(arr, flags=["multi_index"])
    for _ in it:
        idx = it.multi_index
        orig = arr[idx]
        arr[idx] = orig + eps; plus = f()
        arr[idx] = orig - eps; minus = f()
        arr[idx] = orig
        grad[idx] = (plus - minus) / (2 * eps)
    return grad

def forward_loss():
    out = tok(x)
    loss, _ = loss_fn(out)
    return loss

out = tok(x)
_, grad_out = loss_fn(out)
grad_x_analytic = tok.backward(grad_out)
grad_x_numeric = numerical_grad(forward_loss, x)

err = np.max(np.abs(grad_x_analytic - grad_x_numeric) / (np.abs(grad_x_analytic) + np.abs(grad_x_numeric) + 1e-8))
print(f"max relative error: {err:.2e}  ({'OK' if err < 1e-3 else 'FAIL'})")


max relative error: 2.28e-12  (OK)


## 2. Demo Transformer đầy đủ trên dữ liệu TEXT (word-level) — toy sentiment classification

Ở phần 1, `FeatureTokenizer` chỉ dùng được cho dữ liệu bảng SỐ (mỗi cột 1 giá trị liên tục, thứ tự cột không có ý nghĩa). Phần này dùng **`TextClassifierTransformer`** (trong `models/transformer/model.py`) — cùng kiến trúc encoder-decoder đầy đủ (self-attention + cross-attention + feed-forward, mỗi nhánh residual + LayerNorm) như `TabularTransformer`, nhưng tầng tokenize đầu vào đổi thành:

- **`TokenEmbedding`** (`utils/layers/embedding.py`): tra bảng embedding theo CHỈ SỐ TỪ trong từ điển (vocab), thay vì chiếu affine theo giá trị số.
- **`PositionalEncoding`**: cộng thêm thông tin VỊ TRÍ (sin/cos cố định) vào embedding — bắt buộc phải có với text vì thứ tự từ mang ý nghĩa (khác với thứ tự cột trong bảng số, vốn không có ý nghĩa nên `TabularTransformer` không cần).

Bài toán demo: phân loại câu ngắn là "tích cực" hay "tiêu cực" (toy sentiment classification) — encoder học ngữ cảnh giữa các từ trong câu qua self-attention, decoder dùng 1 query token (giống `[CLS]`) cross-attend để gom cả câu thành 1 vector, rồi `Linear + Sigmoid` ra xác suất, huấn luyện bằng `BCELoss` — y hệt luồng huấn luyện đã dùng ở `transformer_demo.ipynb`.

In [ ]:
PAD, UNK = 0, 1

# Toy dataset: câu ngắn gắn nhãn tích cực (1) / tiêu cực (0)
train_sentences = [
    "i love this movie", "this is great", "what a wonderful day",
    "i am very happy", "amazing job team", "this food is delicious",
    "i hate this movie", "this is bad", "what a terrible day",
    "i am very sad", "awful job team", "this food is disgusting",
]
train_labels = np.array([[1], [1], [1], [1], [1], [1], [0], [0], [0], [0], [0], [0]], dtype=float)

# Xây từ điển (vocab) từ toàn bộ tập train: mỗi từ duy nhất -> 1 chỉ số nguyên
# <pad>=0 dùng để đệm các câu ngắn hơn max_len; <unk>=1 dùng cho từ lạ không có trong vocab.
all_words = sorted(set(w for s in train_sentences for w in s.split()))
vocab = {"<pad>": PAD, "<unk>": UNK}
for w in all_words:
    vocab[w] = len(vocab)
vocab_size = len(vocab)
max_len = max(len(s.split()) for s in train_sentences)


def encode(sentence, vocab, max_len):
    """Câu (string) -> list token id, cắt/đệm cho khớp max_len."""
    ids = [vocab.get(w, UNK) for w in sentence.split()]
    ids = ids[:max_len] + [PAD] * max(0, max_len - len(ids))
    return ids


X_train = np.array([encode(s, vocab, max_len) for s in train_sentences])

print("vocab_size:", vocab_size, "(số từ duy nhất + <pad> + <unk>)")
print("max_len:", max_len, "(số từ của câu dài nhất trong tập train)")
print("X_train shape:", X_train.shape)
print(f"Câu đầu tiên: \"{train_sentences[0]}\" -> token ids: {X_train[0]}")

**Đọc kết quả:** `X_train shape (12, 4)` — 12 câu, mỗi câu đệm về đúng 4 token (câu ngắn hơn 4 từ sẽ có `0` (`<pad>`) ở cuối). Mỗi số trong `token ids` là chỉ số của từ đó trong `vocab`, ví dụ câu đầu tiên "i love this movie" (4 từ, không cần đệm) sẽ có 4 chỉ số khác nhau, không trùng `0`.

In [ ]:
from models.transformer.model import TextClassifierTransformer
from utils.loss.BCELoss import BCELoss
from utils.optimizers.Adam import Adam


def make_pad_mask(token_ids):
    # (batch, seq_len) -> (batch, 1, 1, seq_len): 1 ở vị trí từ thật, 0 ở
    # vị trí <pad> -- broadcast được với (batch, heads, seq_q, seq_k) bên
    # trong MultiHeadAttention để chặn attention "chú ý" vào phần đệm.
    return (token_ids != PAD)[:, np.newaxis, np.newaxis, :].astype(float)


np.random.seed(0)
model = TextClassifierTransformer(
    vocab_size=vocab_size, d_model=16, num_heads=2, d_ff=32, num_layers=2, max_len=max_len
)
loss_fn = BCELoss()
optimizer = Adam(model.parameters(), lr=0.01)

train_mask = make_pad_mask(X_train)

n_epochs = 200
loss_history = []
for epoch in range(n_epochs):
    optimizer.zero_grad()

    pred = model(X_train, src_mask=train_mask)
    loss = loss_fn(pred, train_labels)
    loss_history.append(loss)

    grad_loss = loss_fn.backward()
    model.backward(grad_loss)
    optimizer.step()

    if epoch % 40 == 0 or epoch == n_epochs - 1:
        print(f"Epoch {epoch:4d} — loss: {loss:.4f}")

**Đọc kết quả:** với bộ dữ liệu toy chỉ 12 câu và 2 lớp tách biệt rõ ràng (từ vựng "positive" và "negative" gần như không trùng nhau), loss thường giảm rất nhanh về gần 0 (model dễ dàng học thuộc tập train này) — mục tiêu ở đây là kiểm chứng cơ chế (embedding + positional encoding + encoder + decoder + backward) hoạt động đúng đầu-đến-cuối, không phải để đánh giá một benchmark NLP thật.

In [ ]:
# Câu MỚI, chưa từng xuất hiện nguyên văn trong tập train, nhưng ghép từ
# những từ đã có trong vocab -- kiểm tra xem model có khái quát hoá được không.
test_sentences = [
    "i love this food",        # kỳ vọng: tích cực
    "this movie is terrible",  # kỳ vọng: tiêu cực
    "amazing wonderful day",   # kỳ vọng: tích cực
    "this team is bad",        # kỳ vọng: tiêu cực
]
X_test = np.array([encode(s, vocab, max_len) for s in test_sentences])
test_mask = make_pad_mask(X_test)
probs = model(X_test, src_mask=test_mask)

for s, p in zip(test_sentences, probs):
    label = "tích cực" if p[0] >= 0.5 else "tiêu cực"
    print(f"\"{s}\" -> prob={p[0]:.4f} -> {label}")

**Đọc kết quả:** cả 4 câu test đều là tổ hợp MỚI của các từ đã biết (vd "food" chỉ xuất hiện trong câu tích cực lúc train, nhưng "movie"/"team" xuất hiện ở cả 2 lớp) — nếu model dự đoán đúng hướng (câu 1 & 3 xác suất cao gần 1, câu 2 & 4 xác suất thấp gần 0), đó là bằng chứng cho thấy `TextClassifierTransformer` không chỉ "học vẹt" từng câu cụ thể, mà đã học được **embedding của từng từ** (vd nhóm các từ "love/amazing/wonderful/great/delicious" gần nhau trong không gian embedding, tách biệt với nhóm "hate/terrible/bad/awful/disgusting") — đúng tinh thần của self-attention: hiểu ý nghĩa CÂU dựa trên NGỮ CẢNH các từ trong đó, chứ không chỉ khớp mẫu (pattern-matching) trên toàn câu.